# STAT-S681 -- Lecture 3: From Models to Simulators

Runnable lecture-code companion (Python), paired with the "From Models to
Simulators" slides.

Five examples:

1. Bernoulli trials and their sum
2. A two-state hidden Markov model with normal observations
3. A Markov (bigram) language model
4. A topic model
5. A discrete-time SIR epidemic model

Run this notebook top to bottom. No external data files needed, and no
packages beyond NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(681)  # set once, here -- never inside a simulator function


## Example 1: Bernoulli trials and their sum

Model:

$$
X_i \overset{\text{iid}}{\sim} \operatorname{Bernoulli}(p), \quad i = 1, \ldots, n,
\qquad
Y = \sum_{i=1}^n X_i.
$$

- Supplied: `n`, `p`
- Random: $X_1, \ldots, X_n$, and $Y$ ($Y$ is random too -- it's a deterministic *function* of the $X_i$'s, but a deterministic function of a random variable is still random)
- Computed: $Y$, from $X_1, \ldots, X_n$

In [ ]:
n = 20
p = 0.3

# Generate the n individual trials
x = rng.binomial(1, p, size=n)
x


In [ ]:
# The total is a deterministic function of the trials already generated
y = x.sum()
y


In [ ]:
# Inspect what we generated
print(np.bincount(x))
print(x.mean())  # should be near p, but won't equal it exactly


### A second implementation: generate only the total

Same model, shorter code -- this never creates $X_1, \ldots, X_n$.

In [ ]:
y_short = rng.binomial(n, p)
y_short


Check the claim: repeat both implementations many times and compare the resulting distributions.

In [ ]:
n_rep = 2000
y_long_rep = np.array([rng.binomial(1, p, size=n).sum() for _ in range(n_rep)])
y_short_rep = rng.binomial(n, p, size=n_rep)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
bins = np.arange(-0.5, n + 1.5)
axes[0].hist(y_long_rep, bins=bins)
axes[0].set_title("sum of n Bernoullis")
axes[0].set_xlabel("y")
axes[1].hist(y_short_rep, bins=bins)
axes[1].set_title("single Binomial(n, p) draw")
axes[1].set_xlabel("y")
fig.tight_layout()
plt.show()


## Example 2: A two-state hidden Markov model

Model:

$$
Z_1 \sim \operatorname{Categorical}(\pi),
\qquad
Z_t \mid Z_{t-1} \sim \operatorname{Categorical}(A_{Z_{t-1}, \cdot}), \quad t = 2, \ldots, T,
$$

$$
Y_t \mid Z_t = k \sim \operatorname{Normal}(\mu_k, \sigma_k^2).
$$

- Supplied: `pi` (initial-state probabilities), `A` (transition matrix), `mu`, `sigma` (state-specific observation parameters), `T`
- Latent: $Z_1, \ldots, Z_T$
- Observed: $Y_1, \ldots, Y_T$

Concrete story: $Z_t$ is which regime the economy is in this quarter (0 = recession, 1 = expansion), and $Y_t$ is a noisy observed indicator -- quarterly GDP growth (%) -- whose typical level and volatility depend on the regime. We see $Y_t$; the regime itself is never directly observed. This is a real macroeconomics model (Hamilton, 1989).

Written as an explicit loop, with no HMM package, so every random draw is visible.

In [ ]:
def simulate_hmm(n_steps, pi, A, mu, sigma, rng):
    n_states = len(pi)
    z = np.empty(n_steps, dtype=int)
    y = np.empty(n_steps)

    # Initial state: depends only on pi
    z[0] = rng.choice(n_states, p=pi)
    y[0] = rng.normal(mu[z[0]], sigma[z[0]])

    # Every later state depends on the *previous* state;
    # every observation depends on the *current* state.
    for t in range(1, n_steps):
        z[t] = rng.choice(n_states, p=A[z[t - 1], :])
        y[t] = rng.normal(mu[z[t]], sigma[z[t]])

    return z, y


In [ ]:
# Small, concrete parameters: state 0 is "recession", state 1 is
# "expansion", and both regimes are fairly persistent (the diagonal of
# A is large) -- recessions and expansions both tend to last a while.
pi_init = np.array([0.5, 0.5])
A = np.array([
    [0.95, 0.05],   # from recession: stay w.p. .95, shift to expansion w.p. .05
    [0.10, 0.90],   # from expansion: shift to recession w.p. .10, stay w.p. .90
])
mu = np.array([-1.0, 3.0])   # average quarterly GDP growth (%): recession, expansion
sigma = np.array([1.5, 1.0]) # recessions are also more volatile
n_steps = 100

z1, y1 = simulate_hmm(n_steps, pi_init, A, mu, sigma, rng)


In [ ]:
def plot_hmm_run(z, y, title_suffix=""):
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    axes[0].step(np.arange(len(z)), z, where="post")
    axes[0].set_yticks([0, 1])
    axes[0].set_ylabel("regime")
    axes[0].set_title("Latent regime (0 = recession, 1 = expansion) " + title_suffix)
    axes[1].plot(y)
    axes[1].set_ylabel("GDP growth (%)")
    axes[1].set_xlabel("quarter")
    axes[1].set_title("Observed GDP growth " + title_suffix)
    fig.tight_layout()
    plt.show()

plot_hmm_run(z1, y1)


### A second run, same parameters

The rule generating the data (`pi`, `A`, `mu`, `sigma`) is exactly the same as above. Only the realization -- the actual sequence of states and observations -- differs.

In [ ]:
z2, y2 = simulate_hmm(n_steps, pi_init, A, mu, sigma, rng)
plot_hmm_run(z2, y2, title_suffix="(second run, same parameters)")


## Example 3: A Markov (bigram) language model

Model:

$$
P(w_1, \ldots, w_T) = P(w_1) \prod_{t=2}^T P(w_t \mid w_{t-1}).
$$

- Supplied: the transition table $P(w_t \mid w_{t-1})$, estimated below from a short piece of real text
- Random: every $w_t$

We estimate the transition table from actual text (word counts), then treat that table as a *fixed, supplied* parameter and simulate new sequences from it. Today we only supply parameters and simulate; estimating parameters properly from data is Thursday's topic -- this is a preview, not the real thing.

In [ ]:
import re
from collections import defaultdict

# A short public-domain passage (Lewis Carroll, "Alice's Adventures in
# Wonderland," opening lines, 1865) -- real text, not made up.
corpus_text = """Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had
peeped into the book her sister was reading, but it had no pictures or
conversations in it, and what is the use of a book, thought Alice,
without pictures or conversations?"""

# Clean and tokenize: lowercase, strip punctuation, split on whitespace
tokens = re.sub(r"[^\w\s]", "", corpus_text.lower()).split()

vocab = sorted(set(tokens))


In [ ]:
# Count bigram transitions: how often is word j immediately followed
# by word k in the passage?
bigram_counts = defaultdict(lambda: defaultdict(int))
for w_from, w_to in zip(tokens[:-1], tokens[1:]):
    bigram_counts[w_from][w_to] += 1

# Normalize each row into a probability vector: P(. | word) for each
# starting word actually observed as a predecessor.
transition_prob = {}
for w_from, successors in bigram_counts.items():
    total = sum(successors.values())
    transition_prob[w_from] = {w_to: c / total for w_to, c in successors.items()}

# In this short passage, how many distinct successors did each word
# actually get observed with?
n_successors = {w: len(s) for w, s in bigram_counts.items()}
print(sorted(n_successors.values()))
# Most words here were seen with only one successor. This sparsity --
# most contexts observed once or not at all -- is exactly the problem
# that motivates the neural language models on the slides, which share
# parameters across contexts instead of estimating a separate
# distribution for each one.


In [ ]:
def simulate_markov_text(n_tokens, start_word, transition_prob, vocab, rng):
    """current token -> sample next token -> append -> repeat."""
    generated = [start_word]
    for _ in range(n_tokens - 1):
        current = generated[-1]
        successors = transition_prob.get(current)
        if not successors:
            # This word was never seen followed by anything in the
            # passage; restart from a uniformly chosen word rather
            # than getting stuck.
            generated.append(rng.choice(vocab))
        else:
            words, probs = zip(*successors.items())
            generated.append(rng.choice(words, p=probs))
    return generated

print(" ".join(simulate_markov_text(15, "alice", transition_prob, vocab, rng)))
print(" ".join(simulate_markov_text(15, "alice", transition_prob, vocab, rng)))


## Example 4: A topic model

Model:

$$
\theta_d \sim \operatorname{Dirichlet}(\alpha) \quad \text{for each document } d,
$$

$$
z_{d,n} \sim \operatorname{Categorical}(\theta_d),
\qquad
w_{d,n} \sim \operatorname{Categorical}(\beta_{z_{d,n}}).
$$

- Supplied: $\alpha$ (Dirichlet concentration), $\beta$ (topic x word matrices), number of documents, words per document
- Latent: $\theta_d$ (topic proportions), $z_{d,n}$ (topic assignments)
- Observed: $w_{d,n}$ (words)

No topic-model package or real corpus is used here. `beta` is chosen by hand to be clearly interpretable -- one topic is sports-flavored, the other cooking-flavored -- rather than fit to data, so we already know the "ground truth" latent structure the model is generating from.

In [ ]:
vocab_topics = ["ball", "goal", "team", "score", "coach",
                "recipe", "oven", "bake", "kitchen", "dish"]

# Rows are topics, columns are words in vocab_topics; each row sums to 1.
beta = np.array([
    [0.28, 0.24, 0.20, 0.18, 0.10,  0.00, 0.00, 0.00, 0.00, 0.00],  # sports
    [0.00, 0.00, 0.00, 0.00, 0.00,  0.24, 0.22, 0.22, 0.16, 0.16],  # cooking
])
topic_names = ["sports", "cooking"]


def simulate_topic_model(n_docs, n_words_per_doc, alpha, beta, vocab, rng):
    n_topics = beta.shape[0]
    documents = []
    topic_proportions = np.empty((n_docs, n_topics))

    for d in range(n_docs):
        theta_d = rng.dirichlet(alpha)          # this document's topic mixture
        topic_proportions[d, :] = theta_d

        z = rng.choice(n_topics, size=n_words_per_doc, p=theta_d)
        w = [rng.choice(vocab, p=beta[z[n], :]) for n in range(n_words_per_doc)]
        documents.append(w)

    return documents, topic_proportions


alpha = np.array([1.0, 1.0])  # symmetric Dirichlet prior
documents, topic_proportions = simulate_topic_model(
    n_docs=4, n_words_per_doc=12, alpha=alpha, beta=beta,
    vocab=vocab_topics, rng=rng,
)

for d, words in enumerate(documents):
    print(f"Document {d + 1} -- theta:", np.round(topic_proportions[d], 2))
    print(" ", " ".join(words))
    print()


## Example 5: A discrete-time SIR epidemic model

Model, in discrete time steps $t = 0, 1, \ldots, T$, with population size $N = S_t + I_t + R_t$ fixed throughout:

$$
\Delta I_t \mid S_t, I_t \sim \operatorname{Binomial}\!\Big(S_t,\; 1 - (1 - \beta / N)^{I_t}\Big),
\qquad
\Delta R_t \mid I_t \sim \operatorname{Binomial}(I_t, \gamma),
$$

$$
S_{t+1} = S_t - \Delta I_t,
\qquad
I_{t+1} = I_t + \Delta I_t - \Delta R_t,
\qquad
R_{t+1} = R_t + \Delta R_t.
$$

- Supplied: $\beta$, $\gamma$, $S_0$, $I_0$, $R_0$, number of steps $T$
- Random: $\Delta I_t$, $\Delta R_t$ at every step, and $S_t$, $I_t$, $R_t$ themselves (each is a deterministic function of earlier draws, but still random, since those draws are)
- Computed: $S_{t+1}$, $I_{t+1}$, $R_{t+1}$, from $S_t$, $I_t$, $R_t$ and the draws

Every step is exactly two binomial draws -- the same building block as Example 1. $1 - (1 - \beta/N)^{I_t}$ is the probability that a given susceptible individual is infected by at least one of the $I_t$ currently infectious individuals this step, if each infectious individual independently infects each susceptible one with probability $\beta/N$.

In [ ]:
def simulate_sir(n_steps, beta, gamma, S0, I0, R0, rng):
    N = S0 + I0 + R0

    S = np.empty(n_steps + 1, dtype=int)
    I = np.empty(n_steps + 1, dtype=int)
    R = np.empty(n_steps + 1, dtype=int)
    S[0], I[0], R[0] = S0, I0, R0

    for t in range(n_steps):
        p_infect = 1 - (1 - beta / N) ** I[t]
        delta_I = rng.binomial(S[t], p_infect)
        delta_R = rng.binomial(I[t], gamma)

        S[t + 1] = S[t] - delta_I
        I[t + 1] = I[t] + delta_I - delta_R
        R[t + 1] = R[t] + delta_R

    return {"time": np.arange(n_steps + 1), "S": S, "I": I, "R": R}


# An epidemic that takes off: beta / gamma > 1
sir1 = simulate_sir(n_steps=100, beta=0.3, gamma=0.1, S0=995, I0=5, R0=0, rng=rng)

fig, ax = plt.subplots()
ax.plot(sir1["time"], sir1["S"], label="S")
ax.plot(sir1["time"], sir1["I"], label="I")
ax.plot(sir1["time"], sir1["R"], label="R")
ax.set_xlabel("time")
ax.set_ylabel("count")
ax.set_title("Discrete-time SIR (beta = 0.3, gamma = 0.1)")
ax.legend()
plt.show()


### What happens below the epidemic threshold?

Same recovery rate, much lower transmission rate: $\beta / \gamma < 1$.

In [ ]:
sir2 = simulate_sir(n_steps=100, beta=0.08, gamma=0.1, S0=995, I0=5, R0=0, rng=rng)

fig, ax = plt.subplots()
ax.plot(sir2["time"], sir2["I"], color="firebrick")
ax.set_xlabel("time")
ax.set_ylabel("I(t)")
ax.set_title("Same model, beta = 0.08, gamma = 0.1")
plt.show()
